# C1.3 · Red-teaming agents: the injection surface

**Function C — Offensive Security & Research → The Pentester / Red Teamer**  ·  *Security of AI*

---

**Risk.** Retrieval poisoning, tool-output poisoning, multi-turn manipulation.

**Control.** Systematic injection campaigns with measured success rates.

**This lab.** Run a real injection campaign and get a success rate.

| | |
|---|---|
| Open-source tooling | garak, promptfoo |
| Open-weight models | Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C1.3"))

Red-teaming the injection surface: stop asking whether a prompt can be jailbroken and start measuring what fraction of an attack suite reaches a privileged tool.

In [ ]:
from cybercommons import redteam, injection

def target_naive(a):
    if a.surface != redteam.INJECTION:
        return False, "n/a"
    d = injection.Deputy("agent", {"write_file", "http_post"},
                         trust_data_as_instructions=True)
    r = d.handle(a.payload, "write_file", source="document")
    return r["executed"], r["blocked_by"] or "reached the tool"

def target_hardened(a):
    if a.surface != redteam.INJECTION:
        return False, "n/a"
    d = injection.Deputy("agent", {"write_file", "http_post"},
                         trust_data_as_instructions=False)
    r = d.handle(a.payload, "write_file", source="document")
    return r["executed"], r["blocked_by"] or "reached the tool"

inj = [a for a in redteam.SUITE if a.surface == redteam.INJECTION]
for name, t in (("keyword filter only", target_naive), ("provenance enforced", target_hardened)):
    c = redteam.run_campaign(t, name, inj)
    print(name)
    print(c.table())
    print()

The keyword filter blocks the loud attacks and passes the quiet ones. Provenance blocks all of them, because it never asked what the text said.

### Expect

The keyword-filtered target has a non-zero injection ASR — the context-reframe and helpfulness-pretext payloads get through. The provenance-enforced target scores 0.000.

### Your turn

Provenance at 0.000 is suspicious. Construct an attack that defeats it — the payload has to arrive through a channel your system classifies as the principal. That channel is the real finding.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C1.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*